# Exploratory Analysis — Arabic Sign Language Dataset

Inspect classes, image counts, modes, sizes, and view samples.
Production logic lives in the `data_processing/`, `components/`, `model/` and `pipeline/` modules.

In [ ]:
import os, sys, random
# Make project root importable when running from notebooks/
sys.path.append(os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from utils.config import INPUT_DIR, IMAGE_EXTENSIONS, MIN_IMAGES_PER_CLASS
from data_processing.preprocess import preprocess_image

In [ ]:
class_info = []
for dirname, dirnames, filenames in os.walk(INPUT_DIR):
    if len(dirnames) != 0:
        continue
    images = [f for f in filenames if f.lower().endswith(IMAGE_EXTENSIONS)]
    if len(images) <= MIN_IMAGES_PER_CLASS:
        continue
    class_name = os.path.basename(dirname)
    sample_path = os.path.join(dirname, images[0])
    img = Image.open(sample_path)
    class_info.append({
        'class': class_name,
        'count': len(images),
        'mode': img.mode,
        'size': img.size,
    })

df = pd.DataFrame(class_info).sort_values('class').reset_index(drop=True)
print(f'Total classes: {len(df)}')
print(f"Total images: {df['count'].sum()}")
print(f"Unique image modes: {df['mode'].unique()}")
print(f"Unique image sizes: {df['size'].unique()}")
df

In [ ]:
plt.figure(figsize=(16, 5))
sns.barplot(data=df, x='class', y='count', hue='class', palette='crest', legend=False)
plt.title('Images per Arabic Sign Class')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
sample_classes = random.sample(list(df['class']), min(8, len(df)))
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()
for i, cls in enumerate(sample_classes):
    for dirname, dirnames, filenames in os.walk(INPUT_DIR):
        if os.path.basename(dirname) == cls and len(dirnames) == 0:
            img_path = os.path.join(dirname, random.choice(filenames))
            img = Image.open(img_path)
            axes[i].imshow(img, cmap='gray')
            axes[i].set_title(f'{cls}\n{img.mode} {img.size}')
            axes[i].axis('off')
            break
plt.suptitle('Sample Images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Spot-check the preprocessing pipeline on a single image
test_path = None
for dirname, dirnames, filenames in os.walk(INPUT_DIR):
    if len(dirnames) == 0:
        imgs = [f for f in filenames if f.lower().endswith(IMAGE_EXTENSIONS)]
        if imgs:
            test_path = os.path.join(dirname, imgs[0])
            break

result = preprocess_image(test_path)
print('Shape:', result.shape, 'Range:', result.min(), result.max(), 'Dtype:', result.dtype)
plt.figure(figsize=(4,4))
plt.imshow(result[:,:,0], cmap='gray')
plt.title('Preprocessed image')
plt.axis('off')
plt.show()